# Notebook 09 — Evaluation & Baseline Analysis

**Objective.** Measure the quality and the limits of the Phase 1 baseline, component by component, so that Phase 2 starts from evidence rather than assumptions.

**Background.** Notebooks 02 and 03 chose a detector and a tracker without ground truth, from proxy measures. Notebooks 05b and 06 scored the red-light rule and plate reading against labels that the AI assistant made, which no person has checked yet. This notebook adds independent ground truth for detection and tracking, gathers every score in one place with its uncertainty, and ranks what to improve.

**Input.**
- `data/Vehicle_Tracking/`: two clips with ground truth in the MOTChallenge format, a box and an identity for every vehicle in every frame. **VNTraffic** is a junction in Hanoi with dense motorbike traffic (1920×1080, 30 FPS, 501 labelled frames). **AICC22-Custom** is a junction from the AI City Challenge 2022 (1280×720, 10 FPS, 198 labelled frames). Neither was used to choose any setting. Their origin and licence are not recorded here, so they stay local and out of git.
- The labels and outputs of Notebooks 05, 05b, 06 and 08, from their run on the same server.
- `outputs/05b_red_light_violation/review/review_sheet.csv`: a person's verdicts on the red-light labels, used wherever they have been filled in.

**Output** in `outputs/09_baseline_evaluation/` (git-ignored): `evaluation_summary.json` with every score below, and the sheets of error examples.

**Hardware.** Every number here comes from one server, with an NVIDIA L40S and 4 CPU cores. Notebooks 00–08 were committed from a run on an RTX 4090 with 255 cores. Section 7 shows how much the results depend on the hardware.

## 0. Setup

In [1]:
import csv
import json
import logging
import os
import time
from collections import Counter
from pathlib import Path

import boxmot
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

from traffic_violation_detection.detection import DETECTOR_CONFIG, DISPLAY_CONF, detect, load_detector
from traffic_violation_detection.evaluation import (
    apply_review_verdicts,
    average_precision,
    box_iou,
    detection_counts,
    load_mot_ground_truth,
    match_boxes,
    read_review_verdicts,
    tracking_scores,
    wilson_interval,
)
from traffic_violation_detection.plates import mask_plate, score_plate_results
from traffic_violation_detection.red_light import CANDIDATE, match_labels_to_crossings, score_candidates
from traffic_violation_detection.tracking import VEHICLE_CLASSES, update_tracker
from traffic_violation_detection.video import iter_frames, read_frame_at

%config InlineBackend.figure_formats = ["jpeg"]
logging.getLogger("boxmot").setLevel(logging.WARNING)

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(f"No pyproject.toml found in {start} or any parent directory")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA = PROJECT_ROOT / "data"
OUTPUTS = PROJECT_ROOT / "outputs"
OUTPUT_DIR = OUTPUTS / "09_baseline_evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY = {}

GT_ROOT = DATA / "Vehicle_Tracking"
# The frame offset turns a label's frame number into the video's frame index; section 1 checks it.
GT_CLIPS = {
    "VNTraffic": {"video": GT_ROOT / "VNTraffic" / "VNTraffic_Original-video.mp4", "labels": GT_ROOT / "VNTraffic" / "VNTraffic_GroundTruth.txt", "offset": 0},
    "AICC22-Custom": {"video": GT_ROOT / "AICC22-Custom" / "AICC22-Custom_Original-video.mp4", "labels": GT_ROOT / "AICC22-Custom" / "AICC22-Custom_GroundTruth.txt", "offset": -1},
}
for clip in GT_CLIPS.values():
    capture = cv2.VideoCapture(str(clip["video"]))
    clip["fps"], clip["width"], clip["height"] = capture.get(cv2.CAP_PROP_FPS), int(capture.get(3)), int(capture.get(4))
    capture.release()

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none", "| CPU cores:", os.cpu_count())
SUMMARY["hardware"] = {"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "cpu_cores": os.cpu_count()}


def show(image, title=None, width=1400):
    scale = width / image.shape[1]
    image = cv2.resize(image, (width, round(image.shape[0] * scale)), interpolation=cv2.INTER_AREA)
    figure, axis = plt.subplots(figsize=(image.shape[1] / 100, image.shape[0] / 100), dpi=100)
    axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axis.axis("off")
    if title:
        axis.set_title(title, fontsize=11, loc="left")
    plt.show()


def share(part, whole):
    return f"{part}/{whole} = {part / whole:.2f}" if whole else f"{part}/0"


def interval(part, whole):
    low, high = wilson_interval(part, whole)
    return f"{share(part, whole)} (95% interval {low:.2f}-{high:.2f})"

GPU: NVIDIA L40S | CPU cores: 4


## 1. Evaluation data

What can be measured depends on which labels exist and who made them.

In [3]:
detector = load_detector()
detections, detect_ms = {}, {}
for name, clip in GT_CLIPS.items():
    found, seconds = {}, []
    for record in iter_frames(clip["video"]):
        start = time.perf_counter()
        everything = detect(detector, record["image"])
        seconds.append(time.perf_counter() - start)
        found[record["frame_index"]] = [d for d in everything if d["class_id"] in VEHICLE_CLASSES]
    detections[name], detect_ms[name] = found, 1000 * float(np.median(seconds))

# The frame offset that lines the labels up with the video gives the best overlap with the detector's boxes.
truths = {}
print(f"{'clip':<15}" + "".join(f"{f'offset {o:+d}':>11}" for o in range(-2, 3)) + "   (mean IoU of matched boxes)")
for name, clip in GT_CLIPS.items():
    overlap = {o: detection_counts(load_mot_ground_truth(clip["labels"], o), detections[name], 0.1, DISPLAY_CONF)["mean_iou"] for o in range(-2, 3)}
    print(f"{name:<15}" + "".join(f"{overlap[o]:>11.3f}" for o in range(-2, 3)))
    assert max(overlap, key=overlap.get) == clip["offset"]
    truth = load_mot_ground_truth(clip["labels"], clip["offset"])
    truths[name] = {f: objects for f, objects in truth.items() if f in detections[name]}

clip             offset -2  offset -1  offset +0  offset +1  offset +2   (mean IoU of matched boxes)


VNTraffic            0.782      0.822      0.853      0.814      0.772


AICC22-Custom        0.791      0.862      0.794      0.697      0.613


In [4]:
labels = json.loads((DATA / "annotations" / "rlr_occasions_labels.json").read_text())
sheet = OUTPUTS / "05b_red_light_violation" / "review" / "review_sheet.csv"
verdicts = read_review_verdicts(sheet) if sheet.is_file() else {}
labels = apply_review_verdicts(labels, verdicts)
plate_labels_path = DATA / "private_annotations" / "plate_labels.json"
plate_labels = json.loads(plate_labels_path.read_text()) if plate_labels_path.is_file() else None

print(f"{'source':<44}{'used for':<26}{'labelled by':<34}size")
for name, clip in GT_CLIPS.items():
    boxes = sum(len(v) for v in truths[name].values())
    objects = len({o for v in truths[name].values() for o, _ in v})
    print(f"{name + ' ground truth':<44}{'detection, tracking':<26}{'unknown, external':<34}{len(truths[name])} frames, {objects} vehicles, {boxes} boxes")
for group, title in (("clips", "evaluation"), ("development_clips", "development")):
    events = [e for c in labels[group].values() for e in c["events"]]
    print(f"{'red-light labels, ' + title + ' clips':<44}{'red-light rule':<26}{'AI assistant, not reviewed':<34}{len(labels[group])} clips, {dict(Counter(e['label'] for e in events))}")
reviewed = sum(e.get("reviewed_by_a_person", False) for g in ("clips", "development_clips") for c in labels[g].values() for e in c["events"])
print(f"{'review sheet of Notebook 05b':<44}{'replaces red-light labels':<26}{'a person':<34}{reviewed} cases reviewed")
if plate_labels:
    for video, entry in plate_labels["videos"].items():
        tracks = entry["tracks"].values()
        print(f"{'plate labels, ' + video:<44}{'plate reading':<26}{'AI assistant, not reviewed':<34}{sum('text' in t for t in tracks)} readable, {sum('text' not in t and not t.get('unsure') for t in tracks)} unreadable, {sum(bool(t.get('unsure')) for t in tracks)} unsure")
print(f"{'video-test played in reverse':<44}{'wrong-way rule':<26}{'synthetic':<34}see section 4")
SUMMARY["labels"] = {"red_light_cases_reviewed_by_a_person": reviewed, "plate_labels_available": plate_labels is not None}

source                                      used for                  labelled by                       size
VNTraffic ground truth                      detection, tracking       unknown, external                 501 frames, 85 vehicles, 16036 boxes
AICC22-Custom ground truth                  detection, tracking       unknown, external                 198 frames, 21 vehicles, 1185 boxes
red-light labels, evaluation clips          red-light rule            AI assistant, not reviewed        15 clips, {'violation': 14, 'uncertain': 9, 'not_violation': 6}
red-light labels, development clips         red-light rule            AI assistant, not reviewed        3 clips, {'violation': 7, 'not_violation': 2, 'uncertain': 5}
review sheet of Notebook 05b                replaces red-light labels a person                          0 cases reviewed
plate labels, traffic_video_modified        plate reading             AI assistant, not reviewed        24 readable, 2 unreadable, 3 unsure
plate labels, v

In [5]:
frame = read_frame_at(GT_CLIPS["VNTraffic"]["video"], 245)["image"]
for _, box in truths["VNTraffic"][245]:
    cv2.rectangle(frame, tuple(int(v) for v in box[:2]), tuple(int(v) for v in box[2:]), (0, 255, 255), 2)
show(frame, "VNTraffic, frame 245, with its ground truth: every motorbike is labelled with its rider. Local copy only.")

## 2. Object detection

The pipeline keeps detections of five vehicle classes (`bicycle`, `car`, `motorcycle`, `bus`, `truck`) at a confidence of at least 0.10, and shows those above 0.25. A detection is correct when it overlaps a labelled vehicle with an IoU of at least 0.5, each label taken once.

In [6]:
SUMMARY["detection"] = {}
print(f"{'clip':<15}{'confidence':>11}{'IoU':>6}{'TP':>8}{'FP':>7}{'missed':>8}{'precision':>11}{'recall':>8}{'mean IoU':>10}")
for name in GT_CLIPS:
    for confidence in (DETECTOR_CONFIG["conf"], DISPLAY_CONF):
        for iou in (0.5, 0.3):
            c = detection_counts(truths[name], detections[name], iou, confidence)
            print(f"{name:<15}{confidence:>11.2f}{iou:>6.1f}{c['true_positives']:>8}{c['false_positives']:>7}{c['misses']:>8}{c['precision']:>11.3f}{c['recall']:>8.3f}{c['mean_iou']:>10.3f}")
            SUMMARY["detection"][f"{name} conf {confidence} iou {iou}"] = c
ap = {name: {iou: average_precision(truths[name], detections[name], iou) for iou in (0.5, 0.3)} for name in GT_CLIPS}
for name in GT_CLIPS:
    print(f"{name}: average precision {ap[name][0.5]:.3f} at IoU 0.5, {ap[name][0.3]:.3f} at IoU 0.3 | {detect_ms[name]:.1f} ms per frame (median)")
SUMMARY["average_precision"] = ap

clip            confidence   IoU      TP     FP  missed  precision  recall  mean IoU
VNTraffic             0.10   0.5   15646  15489     390      0.503   0.976     0.865
VNTraffic             0.10   0.3   15941  15194      95      0.512   0.994     0.857
VNTraffic             0.25   0.5   15023   5742    1013      0.723   0.937     0.865


VNTraffic             0.25   0.3   15378   5387     658      0.741   0.959     0.855
AICC22-Custom         0.10   0.5    1106   2351      79      0.320   0.933     0.875
AICC22-Custom         0.10   0.3    1141   2316      44      0.330   0.963     0.862
AICC22-Custom         0.25   0.5    1088   1458      97      0.427   0.918     0.878
AICC22-Custom         0.25   0.3    1126   1420      59      0.442   0.950     0.863


VNTraffic: average precision 0.917 at IoU 0.5, 0.940 at IoU 0.3 | 15.3 ms per frame (median)
AICC22-Custom: average precision 0.771 at IoU 0.5, 0.828 at IoU 0.3 | 11.3 ms per frame (median)


**Where the errors are.** Each label is either found or missed, and each detection either matches a label or not. A missed label may still have a box nearby that overlaps it too little, and an unmatched detection may sit on a label that another box took, or far from any label.

In [7]:
def outcomes(truth, found, iou_threshold=0.5, confidence=DISPLAY_CONF):
    # One row per label and one per unmatched detection, with the best overlap each had.
    label_rows, extra_rows = [], []
    for frame, objects in truth.items():
        kept = [d for d in found.get(frame, []) if d["confidence"] >= confidence]
        truth_boxes, boxes = [b for _, b in objects], [d["bbox_xyxy"] for d in kept]
        pairs = {i: (j, v) for i, j, v in match_boxes(truth_boxes, boxes, iou_threshold)}
        overlap = box_iou(truth_boxes, boxes)
        for i, (object_id, box) in enumerate(objects):
            label_rows.append({"frame": frame, "id": object_id, "box": box, "height": box[3] - box[1], "matched": i in pairs,
                               "iou": pairs[i][1] if i in pairs else None, "class": kept[pairs[i][0]]["class_name"] if i in pairs else None,
                               "best_iou": float(overlap[i].max()) if boxes else 0.0})
        used = {j for j, _ in pairs.values()}
        for j, d in enumerate(kept):
            if j not in used:
                extra_rows.append({"frame": frame, "box": d["bbox_xyxy"], "class": d["class_name"], "confidence": d["confidence"],
                                   "best_iou": float(overlap[:, j].max()) if truth_boxes else 0.0})
    return label_rows, extra_rows


BINS = [(0, 32), (32, 64), (64, 128), (128, 10_000)]
rows = {name: outcomes(truths[name], detections[name]) for name in GT_CLIPS}
print("Recall by the height of the labelled box, at confidence 0.25 and IoU 0.5:")
print(f"{'clip':<15}" + "".join(f"{f'{low}-{high if high < 10_000 else chr(8734)} px':>20}" for low, high in BINS))
for name, (label_rows, _) in rows.items():
    cells = []
    for low, high in BINS:
        group = [r for r in label_rows if low <= r["height"] < high]
        cells.append(f"{sum(r['matched'] for r in group)}/{len(group)} = {sum(r['matched'] for r in group) / len(group):.2f}" if group else "-")
    print(f"{name:<15}" + "".join(f"{c:>20}" for c in cells))

print("\nErrors at confidence 0.25 and IoU 0.5:")
for name, (label_rows, extra_rows) in rows.items():
    missed = [r for r in label_rows if not r["matched"]]
    near = sum(r["best_iou"] >= 0.1 for r in missed)
    on_label = sum(r["best_iou"] >= 0.1 for r in extra_rows)
    print(f"  {name:<14} missed {len(missed):>5}: a box nearby but overlapping too little {near:>5}, no box near it {len(missed) - near:>5}")
    print(f"  {'':<14} false positives {len(extra_rows):>5}: on a label already taken or poorly placed {on_label:>5}, far from every label {len(extra_rows) - on_label:>5}")

Recall by the height of the labelled box, at confidence 0.25 and IoU 0.5:
clip                        0-32 px            32-64 px           64-128 px            128-∞ px
VNTraffic               5/29 = 0.17    5233/5861 = 0.89    7901/8253 = 0.96    1884/1893 = 1.00
AICC22-Custom        210/267 = 0.79      518/524 = 0.99      283/290 = 0.98       77/104 = 0.74

Errors at confidence 0.25 and IoU 0.5:
  VNTraffic      missed  1013: a box nearby but overlapping too little   460, no box near it   553
                 false positives  5742: on a label already taken or poorly placed  2219, far from every label  3523
  AICC22-Custom  missed    97: a box nearby but overlapping too little    72, no box near it    25
                 false positives  1458: on a label already taken or poorly placed   127, far from every label  1331


In [8]:
print("Overlap of matched pairs by detected class, at IoU 0.3 (VNTraffic), and the share that would also pass at 0.5:")
label_rows, _ = outcomes(truths["VNTraffic"], detections["VNTraffic"], iou_threshold=0.3)
for class_name in sorted({r["class"] for r in label_rows if r["matched"]}):
    ious = np.array([r["iou"] for r in label_rows if r["class"] == class_name])
    print(f"  {class_name:<11} {len(ious):>6} pairs | median IoU {np.median(ious):.2f} | IoU >= 0.5: {np.mean(ious >= 0.5):.0%}")

Overlap of matched pairs by detected class, at IoU 0.3 (VNTraffic), and the share that would also pass at 0.5:


  bicycle         10 pairs | median IoU 0.85 | IoU >= 0.5: 100%
  bus            492 pairs | median IoU 0.86 | IoU >= 0.5: 99%
  car           6064 pairs | median IoU 0.94 | IoU >= 0.5: 100%
  motorcycle    7147 pairs | median IoU 0.82 | IoU >= 0.5: 95%
  truck         1665 pairs | median IoU 0.94 | IoU >= 0.5: 99%


In [9]:
def crop_sheet(items, title, columns=6, tile=220):
    tiles = []
    for item in items:
        image = read_frame_at(GT_CLIPS[item["clip"]]["video"], item["frame"])["image"]
        x1, y1, x2, y2 = (int(v) for v in item["box"])
        margin = max(20, int(0.4 * max(x2 - x1, y2 - y1)))
        cv2.rectangle(image, (x1, y1), (x2, y2), item.get("color", (0, 0, 255)), 2)
        crop = image[max(0, y1 - margin):y2 + margin, max(0, x1 - margin):x2 + margin]
        crop = cv2.resize(crop, (tile, tile), interpolation=cv2.INTER_AREA)
        cv2.rectangle(crop, (0, 0), (tile, 22), (0, 0, 0), -1)
        cv2.putText(crop, item["caption"], (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)
        tiles.append(crop)
    while len(tiles) % columns:
        tiles.append(np.zeros_like(tiles[0]))
    sheet = np.vstack([np.hstack(tiles[i:i + columns]) for i in range(0, len(tiles), columns)])
    cv2.imwrite(str(OUTPUT_DIR / f"{title}.jpg"), sheet, [cv2.IMWRITE_JPEG_QUALITY, 88])
    show(sheet, title.replace("_", " ") + ". Local copy only.", width=columns * tile)


rng = np.random.default_rng(0)
far = [{**r, "clip": name} for name, (_, extra) in rows.items() for r in extra if r["best_iou"] < 0.1]
sample = [far[i] for i in sorted(rng.choice(len(far), 12, replace=False))]
for n, item in enumerate(sample, start=1):
    item["caption"] = f"FP{n} {item['clip'][:4]} f{item['frame']} {item['class']} {item['confidence']:.2f}"
crop_sheet(sample, "false_positives_far_from_labels")
lost = [{**r, "clip": name, "color": (0, 255, 255)} for name, (label_rows, _) in rows.items() for r in label_rows if not r["matched"] and r["best_iou"] < 0.1]
sample = [lost[i] for i in sorted(rng.choice(len(lost), 12, replace=False))]
for n, item in enumerate(sample, start=1):
    item["caption"] = f"FN{n} {item['clip'][:4]} f{item['frame']} {item['height']:.0f}px high"
crop_sheet(sample, "missed_with_no_box_near")

**Checked by eye** by the AI assistant, from the two sheets above. No person has checked this.

- **Of the 12 detections far from every label, 11 are real vehicles that the labels leave out:** four motorbikes, three cars and a pickup in VNTraffic (FP2–FP9), and a pickup and two cars in AICC22-Custom (FP10–FP12). FP1 is a dark shape cut by the edge of the image, and cannot be judged. So these labels do not cover every vehicle, and **the precision above is only a lower bound.** Most of the 3,523 + 1,331 detections far from every label are probably real vehicles, though only 12 were looked at.
- **Of the 12 labelled vehicles missed with no box near them, 8 are motorbikes in dense groups, partly hidden or blurred** (FN1–FN3, FN7, FN9–FN12, where FN10–FN12 are one rider in three frames). Two are cars hidden behind other objects (FN5, FN8), one is a scooter with its rider standing beside it (FN4), and one is a blurred motorbike cut by the edge of the image (FN6).
- The labelled box of a motorbike includes its rider, but this costs little: 95% of the detector's motorcycle boxes that overlap a label by 0.3 also reach 0.5.

**Alternatives.** Notebook 02 chose YOLO26m at 1280 px from counts, without ground truth. The same comparison against the labels:

In [10]:
ALTERNATIVES = [("yolo26m.pt", 1280), ("yolo26m.pt", 640), ("yolo26s.pt", 1280), ("yolo26l.pt", 1280), ("yolo11m.pt", 1280)]
SUMMARY["detector_alternatives"] = {}
print(f"{'weights':<12}{'size':>6}" + "".join(f"{name[:9] + ' AP50':>16}{'recall':>8}{'precision':>11}" for name in GT_CLIPS) + f"{'ms/frame (1080p)':>18}")
for weights, imgsz in ALTERNATIVES:
    if (weights, imgsz) == (DETECTOR_CONFIG["weights"], DETECTOR_CONFIG["imgsz"]):
        found_by_clip, ms = detections, detect_ms["VNTraffic"]
    else:
        model = load_detector(weights)
        config = {**DETECTOR_CONFIG, "weights": weights, "imgsz": imgsz}
        found_by_clip, ms = {}, None
        for name, clip in GT_CLIPS.items():
            found, seconds = {}, []
            for record in iter_frames(clip["video"]):
                start = time.perf_counter()
                everything = detect(model, record["image"], config)
                seconds.append(time.perf_counter() - start)
                found[record["frame_index"]] = [d for d in everything if d["class_id"] in VEHICLE_CLASSES]
            found_by_clip[name] = found
            if name == "VNTraffic":
                ms = 1000 * float(np.median(seconds))
        del model
        torch.cuda.empty_cache()
    row = {}
    line = f"{weights:<12}{imgsz:>6}"
    for name in GT_CLIPS:
        c = detection_counts(truths[name], found_by_clip[name], 0.5, DISPLAY_CONF)
        row[name] = {"ap50": average_precision(truths[name], found_by_clip[name], 0.5), "recall": c["recall"], "precision": c["precision"]}
        line += f"{row[name]['ap50']:>16.3f}{c['recall']:>8.3f}{c['precision']:>11.3f}"
    row["ms_per_frame"] = ms
    SUMMARY["detector_alternatives"][f"{weights} {imgsz}"] = row
    print(line + f"{ms:>18.1f}" + ("   <- chosen" if (weights, imgsz) == (DETECTOR_CONFIG["weights"], DETECTOR_CONFIG["imgsz"]) else ""))

weights       size  VNTraffic AP50  recall  precision  AICC22-Cu AP50  recall  precision  ms/frame (1080p)


yolo26m.pt    1280           0.917   0.937      0.723           0.771   0.918      0.427              15.3   <- chosen


yolo26m.pt     640           0.919   0.855      0.839           0.782   0.928      0.538              13.6


yolo26s.pt    1280           0.936   0.952      0.754           0.774   0.909      0.479              14.6


yolo26l.pt    1280           0.938   0.964      0.734           0.761   0.925      0.463              19.5


yolo11m.pt    1280           0.943   0.941      0.774           0.772   0.925      0.446              15.0


## 3. Multi-object tracking

The tracker runs on the chosen detector's boxes, as in the pipeline. **MOTA** counts misses, false positives and identity switches against the labels. **IDF1** measures how long each vehicle keeps one identity. A vehicle is **mostly tracked** when it is followed for at least 80% of its labelled frames, and **mostly lost** below 20%.

The variants include the chosen OC-SORT with BoxMOT's default threshold for starting a track, 0.6, which Notebook 05b found to start tracks late in dim footage.

In [11]:
VARIANTS = {
    "ocsort-byte, start 0.6 (chosen)": lambda: boxmot.OcSort(boxmot.OcSortConfig(use_byte=True)),
    "ocsort-byte, start 0.4": lambda: boxmot.OcSort(boxmot.OcSortConfig(use_byte=True, det_thresh=0.4)),
    "ocsort-byte, start 0.25": lambda: boxmot.OcSort(boxmot.OcSortConfig(use_byte=True, det_thresh=0.25)),
    "ocsort, no low-score step": lambda: boxmot.OcSort(),
    "bytetrack": lambda: boxmot.ByteTrack(),
}


def run_tracker(make, found):
    tracker, tracks = make(), {}
    for frame in sorted(found):
        tracks[frame] = [(t["track_id"], t["bbox_xyxy"]) for t in update_tracker(tracker, found[frame], None)]
    return tracks


def first_tracked(truth, tracks, min_frames=10):
    # For each vehicle labelled in at least `min_frames` frames: frames from its first label to its first matching track box.
    first_label, first_track, frames = {}, {}, Counter()
    for frame in sorted(truth):
        objects, hypotheses = truth[frame], tracks.get(frame, [])
        for object_id, _ in objects:
            first_label.setdefault(object_id, frame)
            frames[object_id] += 1
        for i, _, _ in match_boxes([b for _, b in objects], [b for _, b in hypotheses], 0.5):
            first_track.setdefault(objects[i][0], frame)
    eligible = [o for o in first_label if frames[o] >= min_frames]
    return [first_track[o] - first_label[o] for o in eligible if o in first_track], [o for o in eligible if o not in first_track], len(eligible)


tracked_by = {}
SUMMARY["tracking"] = {}
for name, clip in GT_CLIPS.items():
    objects = len({o for v in truths[name].values() for o, _ in v})
    print(f"{name}: {objects} labelled vehicles, {clip['fps']:.0f} FPS")
    print(f"  {'variant':<33}{'MOTA':>6}{'IDF1':>6}{'ID sw.':>8}{'frag.':>7}{'mostly tr.':>11}{'lost':>6}{'recall':>8}{'prec.':>7}{'tracks':>8}{'start delay (s)':>17}{'never':>7}")
    for variant, make in VARIANTS.items():
        tracks = run_tracker(make, detections[name])
        tracked_by[(name, variant)] = tracks
        s = tracking_scores(truths[name], tracks)
        delays, never, eligible = first_tracked(truths[name], tracks)
        ids = len({t for v in tracks.values() for t, _ in v})
        delay = f"{np.median(delays) / clip['fps']:.2f} / {np.percentile(delays, 90) / clip['fps']:.2f}"
        print(f"  {variant:<33}{s['mota']:>6.3f}{s['idf1']:>6.3f}{s['num_switches']:>8}{s['num_fragmentations']:>7}{s['mostly_tracked']:>11}{s['mostly_lost']:>6}{s['recall']:>8.3f}{s['precision']:>7.3f}{ids:>8}{delay:>17}{len(never):>4}/{eligible}")
        SUMMARY["tracking"][f"{name} | {variant}"] = {**s, "track_ids": ids, "median_start_delay_s": float(np.median(delays) / clip["fps"]), "never_tracked": len(never), "eligible": eligible}
print("\nstart delay: median / 90th percentile, from a vehicle's first label to its first matching track box; never: vehicles labelled in 10+ frames that no track matched")

VNTraffic: 85 labelled vehicles, 30 FPS
  variant                            MOTA  IDF1  ID sw.  frag. mostly tr.  lost  recall  prec.  tracks  start delay (s)  never


INFO     OcSort: det_thresh=0.6, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,           
         class_ids=None, asso_func=iou, variable_dt=False

  ocsort-byte, start 0.6 (chosen)   0.644 0.758      40     87         53    15   0.812  0.831      90      0.17 / 2.06   9/82


INFO     OcSort: det_thresh=0.4, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,           
         class_ids=None, asso_func=iou, variable_dt=False

  ocsort-byte, start 0.4            0.550 0.714     112    166         67     5   0.935  0.712     140      0.00 / 0.80   2/82


INFO     OcSort: det_thresh=0.25, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,          
         class_ids=None, asso_func=iou, variable_dt=False

... (11 outputs left out)

INFO     OcSort: det_thresh=0.6, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,           
         class_ids=None, asso_func=iou, variable_dt=False

  ocsort, no low-score step         0.474 0.761       1     17          9     1   0.838  0.698      25      0.00 / 0.50   1/17


INFO     ByteTrack: det_thresh=0.6, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,        
         class_ids=None, asso_func=iou, variable_dt=False

  bytetrack                         0.388 0.757       1      4         17     1   0.952  0.628      24      0.00 / 0.00   1/17

start delay: median / 90th percentile, from a vehicle's first label to its first matching track box; never: vehicles labelled in 10+ frames that no track matched


## 4. Violation detection

**Red light.** Scored as in Notebook 05b, from the crossings saved by its run on this server. Rule version 1 is frozen and holds the official scores. Version 2 was written after its one miss had been seen, so its scores are not a fair test.

In [12]:
MATCH = labels["matching"]
GROUPS = {"evaluation": "clips", "development": "development_clips"}


def score_rule(events_path):
    crossings = json.loads(events_path.read_text())["crossings"]
    scored = {}
    for role, group in GROUPS.items():
        for name, clip in labels[group].items():
            candidates = [r for r in crossings.get(name, []) if r["verdict"] == CANDIDATE]
            scored[name] = (role, score_candidates(candidates, clip["events"], MATCH["max_frame_difference"], MATCH["max_x_difference"]), crossings.get(name, []))
    return scored


RULES = {"version 1 (official)": OUTPUTS / "05b_red_light_violation" / "function_output" / "rlr_occasions_events.json",
         "version 2 (not a fair test)": OUTPUTS / "05b_red_light_violation" / "rule_v2" / "rlr_occasions_events.json"}
scored_rules = {rule: score_rule(path) for rule, path in RULES.items()}
SUMMARY["red_light"] = {}
for rule, scored in scored_rules.items():
    for role in GROUPS:
        outcome = [o for r, o, _ in scored.values() if r == role]
        tp, fp = sum(len(o["true_positives"]) for o in outcome), sum(len(o["false_positives"]) for o in outcome)
        fn, ignored = sum(len(o["false_negatives"]) for o in outcome), sum(len(o["ignored"]) for o in outcome)
        print(f"{rule:<28}{role:<12} TP {tp:>2} FP {fp} ignored {ignored} FN {fn} | precision {interval(tp, tp + fp)} | recall {interval(tp, tp + fn)}")
        SUMMARY["red_light"][f"{rule} {role}"] = {"tp": tp, "fp": fp, "fn": fn, "ignored": ignored}

version 1 (official)        evaluation   TP 13 FP 0 ignored 1 FN 1 | precision 13/13 = 1.00 (95% interval 0.77-1.00) | recall 13/14 = 0.93 (95% interval 0.69-0.99)
version 1 (official)        development  TP  2 FP 0 ignored 0 FN 5 | precision 2/2 = 1.00 (95% interval 0.34-1.00) | recall 2/7 = 0.29 (95% interval 0.08-0.64)
version 2 (not a fair test) evaluation   TP 14 FP 0 ignored 1 FN 0 | precision 14/14 = 1.00 (95% interval 0.78-1.00) | recall 14/14 = 1.00 (95% interval 0.78-1.00)
version 2 (not a fair test) development  TP  2 FP 0 ignored 0 FN 5 | precision 2/2 = 1.00 (95% interval 0.34-1.00) | recall 2/7 = 0.29 (95% interval 0.08-0.64)


In [13]:
print("What rule version 1 did with each labelled violation it missed, and each candidate it should not have made:")
for name, (role, outcome, crossings) in scored_rules["version 1 (official)"].items():
    others = [r for r in crossings if r["verdict"] != CANDIDATE]
    for event in outcome["false_negatives"]:
        record = match_labels_to_crossings([event], others, MATCH["max_frame_difference"], MATCH["max_x_difference"]).get(event["id"])
        print(f"  missed     {event['id']:<6}{role:<12}{record['verdict'] if record else 'no crossing record':<24}{event.get('note', '')[:80]}")
    for candidate, event in outcome["false_positives"]:
        print(f"  extra      clip {name} f{candidate['frame']} x={candidate['x']:.0f} {role:<12}{'label ' + event['id'] + ': ' + event.get('note', '')[:70] if event else 'no label near it'}")

What rule version 1 did with each labelled violation it missed, and each candidate it should not have made:
  missed     M1    evaluation  not_evaluable_ended     Pickup crosses on red about 1 s before the through signal turns green.
  missed     H1    development not_evaluable_signal    Top lamp lit; the image has no colour.
  missed     H2    development not_evaluable_signal    Top lamp lit. Slows near the line, then continues.
  missed     H3    development not_evaluable_signal    Top lamp lit. Continues across the junction.
  missed     H4    development not_evaluable_signal    Top lamp lit. Continues across the junction.
  missed     H5    development no crossing record      Top lamp lit. Continues across the junction.


**Wrong way.** No footage here contains a real wrong-way vehicle, so its recall on real violations cannot be measured. What exists: scenario tests, the real clip, and real tracks played backwards as stand-ins.

In [14]:
evaluation = json.loads((OUTPUTS / "05_violation_detection" / "rule_evaluation.json").read_text())
candidates_file = json.loads((OUTPUTS / "05_violation_detection" / "video-test.wrong_way.candidates.json").read_text())
scenarios = evaluation["scenarios"]
injection = evaluation["reversal_injection"]
caught = injection["caught"] if isinstance(injection["caught"], int) else len(injection["caught"])
eligible = injection["eligible"] if isinstance(injection["eligible"], int) else len(injection["eligible"])
forward = injection["forward_flagged"] if isinstance(injection["forward_flagged"], int) else len(injection["forward_flagged"])
print(f"scenario tests passed: {sum(s['passed'] for s in scenarios)}/{len(scenarios)}")
print(f"real clip video-test: {len(candidates_file['candidates'])} candidates among {sum(candidates_file['status_counts'].values())} tracks, {candidates_file['status_counts']}")
print(f"real tracks reversed: caught {interval(caught, eligible)}; the same tracks forwards flagged: {forward}")
SUMMARY["wrong_way"] = {"scenarios_passed": sum(s["passed"] for s in scenarios), "scenarios": len(scenarios), "real_clip_candidates": len(candidates_file["candidates"]),
                        "reversed_caught": caught, "reversed_eligible": eligible, "status_counts": candidates_file["status_counts"]}

scenario tests passed: 15/15
real clip video-test: 0 candidates among 136 tracks, {'not_in_directional_region': 105, 'no_violation': 20, 'not_evaluable': 11}
real tracks reversed: caught 11/15 = 0.73 (95% interval 0.48-0.89); the same tracks forwards flagged: 0


## 5. License plate recognition

Scored as in Notebook 06, from its saved results. Plate texts are never printed in full.

In [15]:
SUMMARY["plates"] = {}
if plate_labels is None:
    print("The private plate labels are not on this machine, so plate reading is not scored here.")
else:
    for video, path in (("video-test", OUTPUTS / "06_license_plate_recognition" / "video-test" / "video-test_plates.json"),
                        ("traffic_video_modified", OUTPUTS / "06_license_plate_recognition" / "traffic_video_modified" / "traffic_video_modified_plates.json")):
        results = json.loads(path.read_text())["tracks"]
        s = score_plate_results(results, plate_labels["videos"][video]["tracks"])
        role = plate_labels["videos"][video].get("role")
        attempted = s["correct_read"] + s["wrong_read"]
        print(f"{video} ({role}): {s['readable_labels']} plates readable by eye")
        print(f"   read correctly {s['correct_read']}, read wrongly {s['wrong_read']}, right suggestion {s['correct_suggestion']}, wrong suggestion {s['wrong_suggestion']}, declined {s['declined']}, no plate found {s['no_plate']}")
        print(f"   accuracy of texts given as read: {interval(s['correct_read'], attempted)} | readable plates read: {interval(s['correct_read'], s['readable_labels'])}")
        print(f"   plates unreadable by eye: {s['unreadable_labels']}, of which given a text: {s['invented_read'] + s['invented_suggestion']}")
        SUMMARY["plates"][video] = s
    simulated = json.loads((OUTPUTS / "06_license_plate_recognition" / "simulated_red_light" / "plates" / "plates.json").read_text())["events"]
    for event in simulated:
        frame = str(int(event["event_id"].split("-f")[1].split("-")[0]))
        plate = event["license_plate"]
        print(f"simulated red light, candidate at f{frame}: {plate['status']} {mask_plate(plate['text'])}, same as the label: {plate['text'] == plate_labels['simulated_red_light'].get(frame)}")
real = json.loads((OUTPUTS / "06_license_plate_recognition" / "violation_clips" / "plates.json").read_text())
print(f"real red-light candidates (rlr_occasions): {len(real['events'])} clips, statuses {real['status_counts']}")
SUMMARY["plates"]["rlr_occasions"] = real["status_counts"]

video-test (evaluation): 14 plates readable by eye
   read correctly 6, read wrongly 0, right suggestion 3, wrong suggestion 0, declined 3, no plate found 2
   accuracy of texts given as read: 6/6 = 1.00 (95% interval 0.61-1.00) | readable plates read: 6/14 = 0.43 (95% interval 0.21-0.67)
   plates unreadable by eye: 35, of which given a text: 0
traffic_video_modified (development): 24 plates readable by eye
   read correctly 6, read wrongly 0, right suggestion 5, wrong suggestion 5, declined 8, no plate found 0
   accuracy of texts given as read: 6/6 = 1.00 (95% interval 0.61-1.00) | readable plates read: 6/24 = 0.25 (95% interval 0.12-0.45)
   plates unreadable by eye: 2, of which given a text: 0
simulated red light, candidate at f317: read YB****, same as the label: True
simulated red light, candidate at f365: read NN***, same as the label: True
real red-light candidates (rlr_occasions): 17 clips, statuses {'unreadable': 14, 'no_plate_found': 3}


## 6. Processing time and throughput

**The whole pipeline**, from `run_pipeline` in Notebook 08 on this server, with models already loaded:

In [16]:
SUMMARY["pipeline"] = {}
runs = {video: json.loads((OUTPUTS / "08_end_to_end_baseline" / video / f"{video}_summary.json").read_text()) for video in ("traffic_video_modified", "rlr_occasions", "video-test")}
steps = list(runs["rlr_occasions"]["steps_seconds"])
print(f"{'step':<36}" + "".join(f"{video[:16]:>18}" for video in runs))
for step in steps:
    print(f"{step:<36}" + "".join(f"{run['steps_seconds'].get(step, 0):>17.1f}s" for run in runs.values()))
for key, label in (("total_seconds", "total"), ("seconds_of_video", "seconds of video")):
    print(f"{label:<36}" + "".join(f"{run[key]:>17.1f}s" for run in runs.values()))
print(f"{'speed against the video':<36}" + "".join(f"{run['times_real_time']:>17.2f}x" for run in runs.values()))
print(f"{'camera':<36}" + "".join(f"{c:>18}" for c in ("moving", "fixed", "moving")))
SUMMARY["pipeline"] = {video: {k: run[k] for k in ("steps_seconds", "total_seconds", "seconds_of_video", "times_real_time")} for video, run in runs.items()}

step                                  traffic_video_mo     rlr_occasions        video-test
load the detector                                 0.0s              0.0s              0.0s
detect, track and apply the rule                161.3s            217.8s            236.3s
read plates                                      17.6s             48.4s              0.0s
build records and evidence                       15.8s             44.5s              0.0s
review and store                                  0.0s              0.2s              0.0s
write the final video                            36.1s             82.8s              3.4s
total                                           230.8s            393.7s            239.7s
seconds of video                                 27.1s            171.7s             15.0s
speed against the video                          0.12x             0.44x             0.06x
camera                                          moving             fixed            moving

**Per frame on dense traffic.** VNTraffic has about 32 labelled vehicles per frame at 1920×1080. The first 300 frames, with the chosen detector and tracker:

In [17]:
torch.cuda.reset_peak_memory_stats()
tracker = VARIANTS["ocsort-byte, start 0.6 (chosen)"]()
spent = Counter()
frames = iter_frames(GT_CLIPS["VNTraffic"]["video"], 0, 300)
while True:
    t0 = time.perf_counter()
    record = next(frames, None)
    t1 = time.perf_counter()
    if record is None:
        break
    found = detect(detector, record["image"])
    t2 = time.perf_counter()
    update_tracker(tracker, found, record["image"])
    t3 = time.perf_counter()
    spent.update(decode=t1 - t0, detect=t2 - t1, track=t3 - t2)
total = sum(spent.values())
for part, seconds in spent.most_common():
    print(f"  {part:<8}{1000 * seconds / 300:>7.1f} ms per frame  {seconds / total:>5.0%}")
print(f"  {'sum':<8}{1000 * total / 300:>7.1f} ms per frame = {300 / total:.1f} frames per second, against 30 in the video")
print(f"  peak GPU memory of the detector: {torch.cuda.max_memory_allocated() / 2**20:.0f} MiB")
SUMMARY["per_frame_dense_ms"] = {part: 1000 * seconds / 300 for part, seconds in spent.items()}

INFO     OcSort: det_thresh=0.6, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,           
         class_ids=None, asso_func=iou, variable_dt=False

  detect     16.1 ms per frame    74%
  track       4.4 ms per frame    20%
  decode      1.2 ms per frame     6%
  sum        21.7 ms per frame = 46.1 frames per second, against 30 in the video
  peak GPU memory of the detector: 263 MiB


## 7. Dependence on the hardware

The same notebooks, run on two servers, gave slightly different detections. The numbers below compare the committed run of Notebook 05 (RTX 4090) with its run on this server (L40S).

In [18]:
print("wrong-way statuses of the 136 tracks of video-test")
print("  RTX 4090 (committed Notebook 05): {'not_in_directional_region': 106, 'no_violation': 20, 'not_evaluable': 10}")
print(f"  L40S (this server):                {candidates_file['status_counts']}")
(OUTPUT_DIR / "evaluation_summary.json").write_text(json.dumps(SUMMARY, indent=1, default=float))
print(f"\nwrote {(OUTPUT_DIR / 'evaluation_summary.json').relative_to(PROJECT_ROOT)}")

wrong-way statuses of the 136 tracks of video-test
  RTX 4090 (committed Notebook 05): {'not_in_directional_region': 106, 'no_violation': 20, 'not_evaluable': 10}
  L40S (this server):                {'not_in_directional_region': 105, 'no_violation': 20, 'not_evaluable': 11}

wrote outputs/09_baseline_evaluation/evaluation_summary.json


## Results

All numbers come from the outputs above, on the L40S server.

**Performance profile of the baseline.**

| Component | Measure | VNTraffic (Hanoi) | AICC22-Custom | Labels |
|---|---|---|---|---|
| Detection, YOLO26m at 1280 px, confidence 0.25 | recall at IoU 0.5 | 0.937 | 0.918 | external |
| | precision, a lower bound | ≥ 0.723 | ≥ 0.427 | external, incomplete |
| | average precision at IoU 0.5 | 0.917 | 0.771 | external, incomplete |
| Tracking, OC-SORT with BYTE | MOTA / IDF1 | 0.644 / 0.758 | 0.268 / 0.701 | external, incomplete |
| | identity switches | 40, for 85 vehicles | 1, for 21 vehicles | |
| | vehicles never tracked | 9 of 82 | 1 of 17 | |
| | delay before a track starts, median / 90th percentile | 0.17 s / 2.06 s | 0.00 s / 0.50 s | |

| Component | Measure | Value (95% interval) | Labels |
|---|---|---|---|
| Red light, rule version 1, 15 evaluation clips | precision | 13/13 = 1.00 (0.77–1.00) | AI assistant, not reviewed |
| | recall | 13/14 = 0.93 (0.69–0.99) | AI assistant, not reviewed |
| Red light, 3 development clips | recall | 2/7 = 0.29 (0.08–0.64) | not blind |
| Wrong way | real violations | none in the footage, so recall is unknown | |
| | real clip | 0 candidates among 136 tracks | |
| | real tracks played backwards | 11/15 = 0.73 (0.48–0.89) caught | synthetic |
| Plates, video-test | texts given as read that are right | 6/6 = 1.00 (0.61–1.00) | AI assistant, not reviewed |
| | readable plates read | 6/14 = 0.43 (0.21–0.67) | |
| | plates unreadable by eye given a text | 0 of 35 | |
| Plates, real red-light candidates | read | 0 of 17 | |
| Speed, dense 1080p traffic | detection and tracking | 21.7 ms per frame, 46 frames per second | |
| Speed, whole pipeline | against the speed of the video | 0.44× fixed camera, 0.06×–0.12× moving camera | |

**Common error patterns,** most serious first:

1. **Plates cannot be read on real footage.** None of the 17 real candidates, against 6 of 14 readable plates on video-test. In the source, plates are too small and blurred (Notebook 06).
2. **Motorbikes in dense groups are missed.** 8 of the 12 misses checked by eye. Small vehicles are missed most: VNTraffic finds 5 of 29 labels under 32 px high.
3. **Tracks start late, or never.** On VNTraffic, 9 of 82 vehicles never got a track, and for 1 in 10 the track started 2 s or more after the vehicle appeared. A track starts only from a detection of at least 0.6, which is what lost H5 and delayed H2 in Notebook 05b. Lowering that threshold is no cure: at 0.4, vehicles never tracked fall from 9 to 2, but identity switches rise from 40 to 112, and MOTA and IDF1 both fall.
4. **Tracks break.** 40 identity switches on VNTraffic. M1, the one red-light violation that rule version 1 missed, ended as `not_evaluable_ended` because its track broke.
5. **The signal cannot be read on monochrome night footage.** Clip H: 4 of its 5 violations were `not_evaluable_signal`, and the fifth had no track.
6. **Results change slightly with the hardware.** From the RTX 4090 to the L40S, detections moved by about 0.1 px, which changed 1 of 136 wrong-way track statuses, and the reversed-track control from 10 to 11 of 15.

**Bottlenecks.**

- **Accuracy:** the plate reader on real footage, then detection and tracking of dense motorbikes, then reading the signal at night.
- **Speed:** CPU work, not the GPU. Detection takes 16 ms per frame. On a moving camera, camera motion took 171 of 189 ms per frame in Notebook 08 on this 4-core server. On the fixed camera, writing the final video took 83 of 394 s, and the first step also writes an annotated video and the clips.

**Experiments compared.**

| Decision | Chosen | Alternatives measured here | What the labels say |
|---|---|---|---|
| Detector | YOLO26m at 1280 px | YOLO26m at 640 px, YOLO26s, YOLO26l, YOLO11m | 1280 px beats 640 px on recall (0.937 against 0.855 on VNTraffic). YOLO26l finds more (0.964) for 4.2 ms more per frame, and YOLO11m has the highest average precision on VNTraffic (0.943). The choice of Notebook 02 holds against 640 px, but not clearly against the other models. |
| Tracker | OC-SORT with BYTE | ByteTrack, OC-SORT without BYTE, lower start thresholds | ByteTrack scores better on AICC22 (MOTA 0.388, IDF1 0.757) and has fewer switches on VNTraffic (23 against 40), but a lower IDF1 there (0.737 against 0.758). No variant wins on both clips. |
| Red-light rule | version 1 (official) | version 2 | Version 2 adds M1 (14/14), but was written after M1 was seen, so this is not a fair test. |
| Plate thresholds | strict (Notebook 06) | looser settings, Notebook 06 | The strict settings gave no wrong read, at the cost of declining more. |

## Recommendations for Phase 2

In order of priority, each with the evidence behind it:

1. **Get footage and labels fit for the purpose.** That means real violations with sharp plates at least about 35–45 px wide, a person's review of the 30 red-light cases and of the plate labels, and complete ground truth on traffic like VNTraffic. No score above is final without them.
2. **Plate reading:** 0 of 17 on real footage. The source matters before the model, so a camera or region of interest with larger plates comes first. The reader has also never been tested on Vietnamese plates, which a system for Vietnam needs.
3. **Detection of dense motorbikes:** 8 of 12 checked misses. Fine-tune on Vietnamese traffic, and compare YOLO26l and YOLO11m on complete labels.
4. **Tracking:** tune the start threshold together with the detector, rather than alone, as the trade-off above shows, and measure ByteTrack alongside.
5. **Signal state:** take it from the signal controller, or use a colour camera. Monochrome clips cannot be judged.
6. **Speed for a service:** write only each event's clip, not whole annotated videos, and give the moving-camera step more CPU cores or a GPU implementation.
7. **Reproducibility:** keep one reference GPU for scores, and allow for small differences elsewhere.

## Limitations

- **The ground truth covers only two clips** of unknown origin, and does not label every vehicle. Precision and MOTA are therefore lower bounds. Comparisons between variants are fairer than the absolute values, though a variant that finds more unlabelled vehicles still loses.
- **The AICC22 boxes fit less tightly:** mean IoU 0.862 at the best offset. This may explain why 72 of its 97 misses had a box nearby. Its 27 missed vehicles over 128 px high were not examined.
- **The labels for violations and plates were made by the AI assistant** that wrote the code, and no person has reviewed any yet. The review sheet holds 0 verdicts.
- **The samples are small,** so the intervals are wide: 14 labelled red-light violations, and 14 readable plates.
- **The checks by eye** cover 12 false positives and 12 misses, made by the AI assistant.
- **All numbers come from one L40S server with 4 CPU cores,** in single runs, with models already loaded.

## Conclusion

**Completion criterion:** *the baseline has a documented performance profile, and its limitations are supported by observations or measurements rather than assumptions.*

- **Met for the profile.** Every component has a measured score, with its data, its labels and its uncertainty stated.
- **Met for the limitations.** Each error pattern above comes with a count or a measurement. The scores that rest on the assistant's labels remain provisional until a person reviews them.

**Phase 1** is complete against the definition of done in the specification:

- a video is read, detected, tracked and judged by two rules;
- candidates are reviewable records with plate reading attempted;
- review and storage work through a tested interface, though against SQLite rather than a production database;
- the whole chain runs end to end, and its failures and performance are documented here.

**Readiness:** the scoring functions are in `src/traffic_violation_detection/evaluation.py` with tests. To apply a person's review, fill in `your_verdict` in the review sheet and run this notebook again.

**Next steps:** the review service, once the questions about the backend are answered, and the Phase 2 work above, starting with footage and labels.